In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
OUT_ROOT = Path(r"D:\SemanticBiods\data")
LANGUAGES = ["English", "French"]
PERM_RUNS = ["main"]
OUTCOMES = ["RSC", "Turnover"]
FEATURES = ["Local", "Global"]
LAGS = [0, 1, 2, 3, 4, 5]
BASELINE = ["Frequency", "PastChange"]
ALPHAS = np.logspace(-3, 5, 33)
N_OUTER, N_INNER = 5, 4
N_PERM = 100
SEED = 20260901


def standardise(X):
    sd = np.where(X.std(0) < 1e-12, 1.0, X.std(0))
    return (X - X.mean(0)) / sd, X.mean(0), sd


def ridge_path(Z, yc, alphas):
    lam, Q = np.linalg.eigh(Z.T @ Z)
    Qb = Q.T @ (Z.T @ yc)
    return (Q @ (Qb[:, None] / (lam[:, None] + alphas[None, :]))).T


def fit_predict(Xtr, ytr, Xte, alphas):
    Ztr, mean, sd = standardise(Xtr)
    ybar = ytr.mean()
    B = ridge_path(Ztr, ytr - ybar, alphas)
    return ((Xte - mean) / sd) @ B.T + ybar


def nested_cv_r2(X, y, words, alphas):
    oof = np.full(len(y), np.nan)
    for tr, te in GroupKFold(N_OUTER).split(X, y, words):
        sse = np.zeros(len(alphas))
        for itr, ite in GroupKFold(N_INNER).split(X[tr], y[tr], words[tr]):
            pred = fit_predict(X[tr][itr], y[tr][itr], X[tr][ite], alphas)
            sse += ((pred - y[tr][ite][:, None]) ** 2).sum(0)
        oof[te] = fit_predict(X[tr], y[tr], X[te], alphas)[:, int(np.argmin(sse))]
    return float(1 - np.sum((y - oof) ** 2) / np.sum((y - y.mean()) ** 2))


def within_transition_permutation(n, keys, rng):
    """Row permutation that stays inside each decade transition."""
    perm = np.arange(n)
    for value in np.unique(keys):
        idx = np.flatnonzero(keys == value)
        perm[idx] = rng.permutation(idx)
    return perm


def main():
    rows = []
    for language in LANGUAGES:
        for run in PERM_RUNS:
            out = OUT_ROOT / language / run

            for outcome in OUTCOMES:
                df = pd.read_csv(out / f"lagged_{outcome}.csv.gz")
                words = df["word"].to_numpy()
                keys = df["transition_start"].to_numpy()

                y = df["Outcome"].to_numpy(float)
                y = (y - y.mean()) / y.std()

                base_cols = [f"{f}_lag{l}" for f in BASELINE for l in LAGS]
                Xbase = df[base_cols].to_numpy(float)
                base_r2 = nested_cv_r2(Xbase, y, words, ALPHAS)

                for feature in FEATURES:
                    cols = [f"{feature}_lag{l}" for l in LAGS]
                    block = df[cols].to_numpy(float)

                    observed = nested_cv_r2(np.hstack([Xbase, block]), y,
                                            words, ALPHAS) - base_r2

                    rng = np.random.default_rng(
                        SEED + hash((language, outcome, feature)) % 10_000)
                    null = np.empty(N_PERM)
                    t0 = time.time()
                    for b in range(N_PERM):
                        perm = within_transition_permutation(len(df), keys, rng)
                        null[b] = nested_cv_r2(
                            np.hstack([Xbase, block[perm]]), y, words,
                            ALPHAS) - base_r2

                    rows.append(dict(
                        language=language, run=run, outcome=outcome,
                        feature=feature, baseline_R2=base_r2,
                        observed_delta_R2=observed,
                        null_mean=float(null.mean()),
                        null_sd=float(null.std(ddof=1)),
                        null_max=float(null.max()),
                        ratio_to_null_max=float(observed / null.max()),
                        n_perm=N_PERM,
                        p_perm=float((1 + np.sum(null >= observed))
                                     / (N_PERM + 1))))
                    print(f"[{language}/{run}] {outcome} {feature}: "
                          f"obs {observed:.5f}, null max {null.max():.5f}, "
                          f"ratio {observed / null.max():.1f}  "
                          f"({(time.time() - t0) / 60:.1f} min)", flush=True)

    tab = OUT_ROOT / "tables"
    tab.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(tab / "permutation_floor.csv", index=False)
    print("\nwrote tables/permutation_floor.csv")


if __name__ == "__main__":
    main()


[English/main] RSC Local: obs 0.01665, null max 0.00014, ratio 119.7  (2.1 min)
[English/main] RSC Global: obs 0.00137, null max 0.00177, ratio 0.8  (1.6 min)
[English/main] Turnover Local: obs 0.01532, null max 0.00010, ratio 154.3  (1.8 min)
[English/main] Turnover Global: obs 0.00021, null max 0.00100, ratio 0.2  (2.0 min)
[French/main] RSC Local: obs 0.00905, null max 0.00019, ratio 48.4  (1.7 min)
[French/main] RSC Global: obs 0.00124, null max 0.00041, ratio 3.0  (1.7 min)
[French/main] Turnover Local: obs 0.00873, null max 0.00048, ratio 18.1  (1.6 min)
[French/main] Turnover Global: obs 0.00080, null max 0.00129, ratio 0.6  (1.5 min)

wrote tables/permutation_floor.csv


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
OUT_ROOT = Path(r"D:\SemanticBiods\OSF")
LANGUAGES = ["English", "French"]
RUNS = ["main", "window"]
OUTCOMES = ["RSC", "Turnover"]
FEATURES = ["Local", "Global"]
LAGS = [0, 1, 2, 3, 4, 5]
BASELINE = ["Frequency", "PastChange"]
ALPHAS = np.logspace(-3, 5, 33)
N_OUTER, N_INNER = 5, 4

def standardise(X):
    sd = np.where(X.std(0) < 1e-12, 1.0, X.std(0))
    return (X - X.mean(0)) / sd, X.mean(0), sd


def ridge_path(Z, yc, alphas):
    lam, Q = np.linalg.eigh(Z.T @ Z)
    Qb = Q.T @ (Z.T @ yc)
    return (Q @ (Qb[:, None] / (lam[:, None] + alphas[None, :]))).T


def fit_predict(Xtr, ytr, Xte, alphas):
    Ztr, mean, sd = standardise(Xtr)
    ybar = ytr.mean()
    B = ridge_path(Ztr, ytr - ybar, alphas)
    return ((Xte - mean) / sd) @ B.T + ybar


def nested_cv_r2(X, y, words, alphas):
    oof = np.full(len(y), np.nan)
    for tr, te in GroupKFold(N_OUTER).split(X, y, words):
        sse = np.zeros(len(alphas))
        for itr, ite in GroupKFold(N_INNER).split(X[tr], y[tr], words[tr]):
            pred = fit_predict(X[tr][itr], y[tr][itr], X[tr][ite], alphas)
            sse += ((pred - y[tr][ite][:, None]) ** 2).sum(0)
        oof[te] = fit_predict(X[tr], y[tr], X[te], alphas)[:, int(np.argmin(sse))]
    return float(1 - np.sum((y - oof) ** 2) / np.sum((y - y.mean()) ** 2))


def main():
    rows = []
    for language in LANGUAGES:
        for run in RUNS:
            out = OUT_ROOT / language / run

            for outcome in OUTCOMES:
                df = pd.read_csv(out / f"lagged_{outcome}.csv.gz")
                words = df["word"].to_numpy()
                y = df["Outcome"].to_numpy(float)
                y = (y - y.mean()) / y.std()

                base_cols = [f"{f}_lag{l}" for f in BASELINE for l in LAGS]
                Xbase = df[base_cols].to_numpy(float)
                base_r2 = nested_cv_r2(Xbase, y, words, ALPHAS)

                for feature in FEATURES:
                    lag0 = df[f"{feature}_lag0"].to_numpy(float)
                    lag1 = df[f"{feature}_lag1"].to_numpy(float)
                    free = df[[f"{feature}_lag{l}" for l in LAGS]].to_numpy(float)

                    variants = {
                        "level": lag0[:, None],
                        "change": (lag0 - lag1)[:, None],
                        "free": free,
                    }
                    gains = {}
                    for name, block in variants.items():
                        gains[name] = nested_cv_r2(
                            np.hstack([Xbase, block]), y, words,
                            ALPHAS) - base_r2

                    for name, gain in gains.items():
                        rows.append(dict(
                            language=language, run=run, outcome=outcome,
                            feature=feature, parametrisation=name,
                            baseline_R2=base_r2, delta_R2=gain,
                            share_of_free=(gain / gains["free"]
                                           if gains["free"] > 0 else np.nan)))
                    print(f"[{language}/{run}] {outcome} {feature}: "
                          f"level {gains['level']:.5f}  "
                          f"change {gains['change']:.5f}  "
                          f"free {gains['free']:.5f}", flush=True)

    long = pd.DataFrame(rows)
    tab = OUT_ROOT / "tables"
    tab.mkdir(parents=True, exist_ok=True)
    long.to_csv(tab / "level_vs_change.csv", index=False)

    wide = long.pivot_table(
        index=["language", "run", "outcome", "feature"],
        columns="parametrisation", values=["delta_R2", "share_of_free"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide.reset_index().to_csv(tab / "level_vs_change_wide.csv", index=False)
    print("\nwrote tables/level_vs_change.csv")


if __name__ == "__main__":
    main()


[English/main] RSC Local: level 0.00367  change 0.01218  free 0.01665
[English/main] RSC Global: level 0.00065  change 0.00064  free 0.00137
[English/main] Turnover Local: level 0.01023  change 0.00684  free 0.01532
[English/main] Turnover Global: level 0.00010  change 0.00000  free 0.00021
[English/window] RSC Local: level 0.00395  change 0.01412  free 0.02020
[English/window] RSC Global: level 0.00077  change 0.00081  free 0.00179
[English/window] Turnover Local: level 0.01243  change 0.00756  free 0.01841
[English/window] Turnover Global: level -0.00001  change -0.00002  free 0.00008
[French/main] RSC Local: level 0.00228  change 0.00707  free 0.00905
[French/main] RSC Global: level 0.00073  change 0.00034  free 0.00124
[French/main] Turnover Local: level 0.00342  change 0.00462  free 0.00873
[French/main] Turnover Global: level 0.00010  change -0.00000  free 0.00080
[French/window] RSC Local: level 0.00140  change 0.00606  free 0.00887
[French/window] RSC Global: level 0.00056  cha

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# ===========================================================================
# CONFIGURATION
# ===========================================================================

OUT_ROOT = Path(r"D:\SemanticBiods\data")

LANGUAGES = ["English", "French"]
RUNS = ["main", "window"]
OUTCOMES = ["RSC", "Turnover"]
FEATURES = ["Local", "Global"]

LAGS = [0, 1, 2, 3, 4, 5]
BASELINE = ["Frequency", "PastChange"]

ALPHAS = np.logspace(-3, 5, 33)
N_OUTER, N_INNER = 5, 4

# ===========================================================================


def standardise(X):
    sd = np.where(X.std(0) < 1e-12, 1.0, X.std(0))
    return (X - X.mean(0)) / sd, X.mean(0), sd


def ridge_path(Z, yc, alphas):
    lam, Q = np.linalg.eigh(Z.T @ Z)
    Qb = Q.T @ (Z.T @ yc)
    return (Q @ (Qb[:, None] / (lam[:, None] + alphas[None, :]))).T


def fit_predict(Xtr, ytr, Xte, alphas):
    Ztr, mean, sd = standardise(Xtr)
    ybar = ytr.mean()
    B = ridge_path(Ztr, ytr - ybar, alphas)
    return ((Xte - mean) / sd) @ B.T + ybar


def nested_cv_r2(X, y, words, alphas):
    oof = np.full(len(y), np.nan)
    for tr, te in GroupKFold(N_OUTER).split(X, y, words):
        sse = np.zeros(len(alphas))
        for itr, ite in GroupKFold(N_INNER).split(X[tr], y[tr], words[tr]):
            pred = fit_predict(X[tr][itr], y[tr][itr], X[tr][ite], alphas)
            sse += ((pred - y[tr][ite][:, None]) ** 2).sum(0)
        oof[te] = fit_predict(X[tr], y[tr], X[te], alphas)[:, int(np.argmin(sse))]
    return float(1 - np.sum((y - oof) ** 2) / np.sum((y - y.mean()) ** 2))


def main():
    rows = []
    for language in LANGUAGES:
        for run in RUNS:
            out = OUT_ROOT / language / run

            for outcome in OUTCOMES:
                df = pd.read_csv(out / f"lagged_{outcome}.csv.gz")
                words = df["word"].to_numpy()
                y = df["Outcome"].to_numpy(float)
                y = (y - y.mean()) / y.std()

                base_cols = [f"{f}_lag{l}" for f in BASELINE for l in LAGS]
                Xbase = df[base_cols].to_numpy(float)
                base_r2 = nested_cv_r2(Xbase, y, words, ALPHAS)

                for feature in FEATURES:
                    lag0 = df[f"{feature}_lag0"].to_numpy(float)
                    lag1 = df[f"{feature}_lag1"].to_numpy(float)
                    free = df[[f"{feature}_lag{l}" for l in LAGS]].to_numpy(float)

                    variants = {
                        "level": lag0[:, None],
                        "change": (lag0 - lag1)[:, None],
                        "free": free,
                    }
                    gains = {}
                    for name, block in variants.items():
                        gains[name] = nested_cv_r2(
                            np.hstack([Xbase, block]), y, words,
                            ALPHAS) - base_r2

                    for name, gain in gains.items():
                        rows.append(dict(
                            language=language, run=run, outcome=outcome,
                            feature=feature, parametrisation=name,
                            baseline_R2=base_r2, delta_R2=gain,
                            share_of_free=(gain / gains["free"]
                                           if gains["free"] > 0 else np.nan)))
                    print(f"[{language}/{run}] {outcome} {feature}: "
                          f"level {gains['level']:.5f}  "
                          f"change {gains['change']:.5f}  "
                          f"free {gains['free']:.5f}", flush=True)

    long = pd.DataFrame(rows)
    tab = OUT_ROOT / "tables"
    tab.mkdir(parents=True, exist_ok=True)
    long.to_csv(tab / "level_vs_change.csv", index=False)

    wide = long.pivot_table(
        index=["language", "run", "outcome", "feature"],
        columns="parametrisation", values=["delta_R2", "share_of_free"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide.reset_index().to_csv(tab / "level_vs_change_wide.csv", index=False)
    print("\nwrote tables/level_vs_change.csv")


if __name__ == "__main__":
    main()


[English/main] RSC Local: level 0.00367  change 0.01218  free 0.01665
[English/main] RSC Global: level 0.00065  change 0.00064  free 0.00137
[English/main] Turnover Local: level 0.01023  change 0.00684  free 0.01532
[English/main] Turnover Global: level 0.00010  change 0.00000  free 0.00021
[English/window] RSC Local: level 0.00395  change 0.01412  free 0.02020
[English/window] RSC Global: level 0.00077  change 0.00081  free 0.00179
[English/window] Turnover Local: level 0.01243  change 0.00756  free 0.01841
[English/window] Turnover Global: level -0.00001  change -0.00002  free 0.00008
[French/main] RSC Local: level 0.00228  change 0.00707  free 0.00905
[French/main] RSC Global: level 0.00073  change 0.00034  free 0.00124
[French/main] Turnover Local: level 0.00342  change 0.00462  free 0.00873
[French/main] Turnover Global: level 0.00010  change -0.00000  free 0.00080
[French/window] RSC Local: level 0.00140  change 0.00606  free 0.00887
[French/window] RSC Global: level 0.00056  cha